# Calls


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
Calls = pd.read_excel('/content/drive/MyDrive/fin_project/not clean/Calls.xlsx',dtype={'Id': str, 'CONTACTID': str})
print(Calls.shape)
print()
summary_calls = pd.DataFrame({
    'dtype': Calls.dtypes,
    'non_null': Calls.notna().sum(),
    'null': Calls.isna().sum(),
    'null_%': (Calls.isna().sum() / len(Calls) * 100).round(1),
    'unique': Calls.nunique()
})
print(summary_calls)

In [ ]:
# Drop columns that are 100% empty
Calls = Calls.drop(columns=['Dialled Number', 'Tag'])
print("Dropped columns: Dialled Number, Tag")

# Look at categorical columns
print("\nCall Type:")
print(Calls['Call Type'].value_counts())

print("\nCall Status:")
print(Calls['Call Status'].value_counts())

print("\nOutgoing Call Status:")
print(Calls['Outgoing Call Status'].value_counts(dropna=False))

print("\nScheduled in CRM:")
print(Calls['Scheduled in CRM'].value_counts(dropna=False))

# Calls start time


In [ ]:
# Check: NaN in Outgoing Call Status = non-Outbound calls?
mask_not_outbound = Calls['Call Type'] != 'Outbound'
print(f"Non-Outbound calls: {mask_not_outbound.sum()}")
print(f"NaN in Outgoing Call Status: {Calls['Outgoing Call Status'].isna().sum()}")

# Convert data types
# Call Start Time -> datetime
Calls['Call Start Time'] = pd.to_datetime(Calls['Call Start Time'])

# CONTACTID -> string (for joining with Contacts)
Calls['CONTACTID'] = Calls['CONTACTID'].astype('Int64').astype(str).replace('<NA>', None)

# Scheduled in CRM -> boolean type
Calls['Scheduled in CRM'] = Calls['Scheduled in CRM'].map({0.0: False, 1.0: True})

print("\nTypes after conversion:")
print(Calls[['Call Start Time', 'CONTACTID', 'Scheduled in CRM']].dtypes)

In [ ]:
# Call Start Time -> datetime
Calls['Call Start Time'] = pd.to_datetime(
    Calls['Call Start Time'], format='%d.%m.%Y %H:%M'
)
print(Calls['Call Start Time'].dtype)
print(Calls['Call Start Time'].head(3))

# Contact ID


In [ ]:
# CONTACTID -> string for joining
Calls['CONTACTID'] = Calls['CONTACTID'].astype('Int64').astype(str).replace('<NA>', None)
print(Calls['CONTACTID'].dtype)
print(Calls['CONTACTID'].value_counts(dropna=False).head(5))

# Scheduled in CRM


In [ ]:
# Scheduled in CRM -> nullable boolean (0=False, 1=True, NaN=NaN)
Calls['Scheduled in CRM'] = Calls['Scheduled in CRM'].map(
    {0.0: False, 1.0: True}
).astype('boolean')
print(Calls['Scheduled in CRM'].dtype)
print(Calls['Scheduled in CRM'].value_counts(dropna=False))

# Call Duration


In [ ]:
print(Calls['Call Duration (in seconds)'].describe())
print()
print(f"Zero values: {(Calls['Call Duration (in seconds)'] == 0).sum()}")
print(f"Negative values: {(Calls['Call Duration (in seconds)'] < 0).sum()}")
print(f"Missing values: {Calls['Call Duration (in seconds)'].isna().sum()}")

In [ ]:
print(Calls.loc[Calls['Call Duration (in seconds)'] == 0, 'Call Status'].value_counts())
print()
print(Calls.loc[Calls['Call Duration (in seconds)'] == 0, 'Call Type'].value_counts())

# Saving


In [ ]:
Calls.to_parquet('/content/drive/MyDrive/fin_project/Calls_clean.parquet', index=False)
print("Calls saved!")

# Descriptive statistics


In [ ]:
# Check for duplicates in Calls
print(f"Duplicates across all columns: {Calls.duplicated().sum()}")
print(f"Duplicates by Id: {Calls.duplicated(subset=['Id']).sum()}")

In [ ]:
# General information
print(f"Total calls: {len(Calls)}")
print(f"Period: {Calls['Call Start Time'].min()} — {Calls['Call Start Time'].max()}")
print(f"Days in period: {(Calls['Call Start Time'].max() - Calls['Call Start Time'].min()).days}")
print(f"Unique managers: {Calls['Call Owner Name'].nunique()}")
print()

# Numeric columns
print("Call Duration (in seconds):")
print(f"  Mean:    {Calls['Call Duration (in seconds)'].mean():.1f} sec ({Calls['Call Duration (in seconds)'].mean()/60:.1f} min)")
print(f"  Median:  {Calls['Call Duration (in seconds)'].median():.1f} sec ({Calls['Call Duration (in seconds)'].median()/60:.1f} min)")
print(f"  Mode:    {Calls['Call Duration (in seconds)'].mode()[0]:.1f} sec")
print(f"  Max:     {Calls['Call Duration (in seconds)'].max():.1f} sec ({Calls['Call Duration (in seconds)'].max()/60:.1f} min)")
print(f"  Missing: {Calls['Call Duration (in seconds)'].isna().sum()}")

In [ ]:
# Категориальные столбцы
print("Call Type:")
print(Calls['Call Type'].value_counts())
print()
print("Call Status:")
print(Calls['Call Status'].value_counts())
print()
print("Outgoing Call Status:")
print(Calls['Outgoing Call Status'].value_counts(dropna=False))
print()
print("Scheduled in CRM:")
print(Calls['Scheduled in CRM'].value_counts(dropna=False))

In [ ]:
# Monthly call trend
Calls['Call Month'] = Calls['Call Start Time'].dt.to_period('M')
print("Calls by month:")
print(Calls['Call Month'].value_counts().sort_index())
print()

# Top 10 managers by number of calls
print("Top 10 managers by calls:")
print(Calls['Call Owner Name'].value_counts().head(10))

In [ ]:
# Answered calls only (Duration > 0)
answered = Calls[Calls['Call Duration (in seconds)'] > 0]
print(f"Answered calls: {len(answered)} ({len(answered)/len(Calls)*100:.1f}%)")
print()
print(f"  Mean:    {answered['Call Duration (in seconds)'].mean():.1f} sec ({answered['Call Duration (in seconds)'].mean()/60:.1f} min)")
print(f"  Median:  {answered['Call Duration (in seconds)'].median():.1f} sec ({answered['Call Duration (in seconds)'].median()/60:.1f} min)")
print(f"  Max:     {answered['Call Duration (in seconds)'].max():.1f} sec ({answered['Call Duration (in seconds)'].max()/60:.1f} min)")
print()

# Call conversion
print("Conversion by Call Status:")
total = len(Calls)
for status, count in Calls['Call Status'].value_counts().items():
    print(f"  {status}: {count} ({count/total*100:.1f}%)")